Load train and test datasets

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.preprocessing import preprocess_complaint

In [2]:
train_df = pd.read_parquet('../data/processed/train_complaints.parquet')
test_df = pd.read_parquet('../data/processed/test_complaints.parquet')

In [3]:
print(train_df.shape)
print(test_df.shape)

(812720, 4)
(203180, 4)


In [4]:
train_df.head()

,Complaint ID,Consumer complaint narrative,Department,Priority
0,7251882,This is regarding the Texas B-On-Time Student ...,Student loan,high_priority
1,16800276,"On top of other discrepancies, XXXX XXXX accou...",Credit reporting,standard
2,2292018,There was fraudulent activity on my account wh...,Bank accounts,high_priority
3,7835057,I have consistently ensured my payments are ma...,Card services,standard
4,4313529,I tried to send them a letter last XX/XX/2020....,Mortgage,standard


Define features and targets

In [5]:
X_train = train_df["Consumer complaint narrative"]
X_test  = test_df["Consumer complaint narrative"]

y_train_dept = train_df["Department"]
y_test_dept  = test_df["Department"]

y_train_priority = train_df["Priority"]
y_test_priority  = test_df["Priority"]

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

model_name = "distilbert-base-uncased"

# ---- APPLY PREPROCESS FUNCTION ----
X_train_clean = [preprocess_complaint(t) for t in X_train]
X_test_clean  = [preprocess_complaint(t) for t in X_test]

# ---- TOKENIZER ----
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

args = TrainingArguments(
    output_dir="./model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",           
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1"
)

# ---- DEPARTMENT ----
train_ds_dept = Dataset.from_dict({"text": X_train_clean, "label": y_train_dept})
test_ds_dept  = Dataset.from_dict({"text": X_test_clean,  "label": y_test_dept})

train_ds_dept = train_ds_dept.map(tokenize, batched=True)
test_ds_dept  = test_ds_dept.map(tokenize, batched=True)

train_ds_dept.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds_dept.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

model_dept = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(set(y_train_dept))
)

trainer_dept = Trainer(
    model=model_dept,
    args=args,
    train_dataset=train_ds_dept,
    eval_dataset=test_ds_dept,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer_dept.train()

y_pred_dept = np.argmax(trainer_dept.predict(test_ds_dept).predictions, axis=1)
print("DEPARTMENT REPORT\n", classification_report(y_test_dept, y_pred_dept))
print("DEPARTMENT MACRO F1:", f1_score(y_test_dept, y_pred_dept, average="macro"))

# ---- PRIORITY ----
train_ds_pri = Dataset.from_dict({"text": X_train_clean, "label": y_train_priority})
test_ds_pri  = Dataset.from_dict({"text": X_test_clean,  "label": y_test_priority})

train_ds_pri = train_ds_pri.map(tokenize, batched=True)
test_ds_pri  = test_ds_pri.map(tokenize, batched=True)

train_ds_pri.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds_pri.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

model_pri = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(set(y_train_priority))
)

trainer_pri = Trainer(
    model=model_pri,
    args=args,
    train_dataset=train_ds_pri,
    eval_dataset=test_ds_pri,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer_pri.train()

y_pred_pri = np.argmax(trainer_pri.predict(test_ds_pri).predictions, axis=1)
print("\nPRIORITY REPORT\n", classification_report(y_test_priority, y_pred_pri))
print("PRIORITY MACRO F1:", f1_score(y_test_priority, y_pred_pri, average="macro"))